# Django Framework: Complete Structured Guide

**With ASGI/WSGI, Request-Response Flows, Full Applications & REST APIs**

---
## Module 1: Understanding Web Frameworks
---

### 1.1 What is a Web Framework?

**Definition:**
A web framework is a collection of pre-written code, libraries, tools, and conventions that provide a structured foundation for building web applications.

### 1.2 Key Components of a Framework

| Component | Purpose |
|-----------|----------|
| **Routing** | Maps URLs to views |
| **ORM** | Database abstraction |
| **Templating** | Dynamic HTML |
| **Middleware** | Request processing |
| **Forms** | Input validation |
| **Authentication** | User login |
| **Admin** | CRUD interface |
| **Testing** | Unit tests |

---
## Module 2: Web Server Protocols - WSGI & ASGI
---

### 2.1 What is WSGI?

**WSGI** = Web Server Gateway Interface

**Definition:** A standard interface between web servers and Python web applications.

**How it works:**
- Web server receives HTTP request
- Converts to WSGI environ dictionary
- Calls application (Django) with environ + start_response
- Application returns response
- Web server sends to client

### 2.2 What is ASGI?

**ASGI** = Asynchronous Server Gateway Interface

**Definition:** Modern evolution of WSGI supporting asynchronous operations.

**Key Differences:**

| Aspect | WSGI | ASGI |
|--------|------|------|
| **Type** | Synchronous | Asynchronous |
| **WebSocket** | ❌ No | ✅ Yes |
| **Streaming** | Limited | ✅ Full support |
| **Real-time** | ❌ No | ✅ Yes |
| **Server** | Gunicorn, uWSGI | Daphne, Uvicorn |
| **Use Case** | Regular web apps | Real-time, APIs |
| **Performance** | Good | Excellent |

In [1]:
# WSGI vs ASGI Comparison

print("WSGI (Traditional Django Deployment)")
print("="*50)
wsgi = """
Web Server (Nginx, Apache)
        ↓
WSGI Application Server (Gunicorn, uWSGI)
        ↓
Django Application
        ↓
Database

Characteristics:
- One request at a time per worker
- Multiple workers handle concurrent requests
- No WebSocket support
- Blocking I/O

# wsgi.py
import os
from django.core.wsgi import get_wsgi_application
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'myproject.settings')
application = get_wsgi_application()

# Run: gunicorn myproject.wsgi:application
"""
print(wsgi)

print("\n\nASGI (Modern Django Deployment)")
print("="*50)
asgi = """
Web Server (Nginx, Apache)
        ↓
ASGI Application Server (Daphne, Uvicorn)
        ↓
Django Application (Async Views)
        ↓
Database / External Services

Characteristics:
- Handles multiple requests simultaneously
- WebSocket support for real-time
- Non-blocking I/O
- Better resource utilization

# asgi.py
import os
from django.core.asgi import get_asgi_application
from channels.routing import ProtocolTypeRouter, URLRouter

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'myproject.settings')

application = ProtocolTypeRouter({
    'http': get_asgi_application(),
    'websocket': URLRouter([
        path('ws/chat/<room_name>/', consumers.ChatConsumer.as_asgi()),
    ]),
})

# Run: daphne myproject.asgi:application
"""
print(asgi)

WSGI (Traditional Django Deployment)

Web Server (Nginx, Apache)
        ↓
WSGI Application Server (Gunicorn, uWSGI)
        ↓
Django Application
        ↓
Database

Characteristics:
- One request at a time per worker
- Multiple workers handle concurrent requests
- No WebSocket support
- Blocking I/O

# wsgi.py
import os
from django.core.wsgi import get_wsgi_application
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'myproject.settings')
application = get_wsgi_application()

# Run: gunicorn myproject.wsgi:application



ASGI (Modern Django Deployment)

Web Server (Nginx, Apache)
        ↓
ASGI Application Server (Daphne, Uvicorn)
        ↓
Django Application (Async Views)
        ↓
Database / External Services

Characteristics:
- Handles multiple requests simultaneously
- WebSocket support for real-time
- Non-blocking I/O
- Better resource utilization

# asgi.py
import os
from django.core.asgi import get_asgi_application
from channels.routing import ProtocolTypeRouter, URLRouter

os.e

---
## Module 3: Django Request-Response Flow
---

### 3.1 Complete Request-Response Cycle

```
┌─────────────────────────────────────────────────────────────────┐
│                    DJANGO REQUEST-RESPONSE FLOW                  │
└─────────────────────────────────────────────────────────────────┘

1. USER SENDS REQUEST
   ┌──────────────────────────────────────────────┐
   │  Browser: GET /blog/post/django-tutorial/    │
   └──────────────────────────────────────────────┘
                    ↓

2. WEB SERVER RECEIVES
   ┌──────────────────────────────────────────────┐
   │  Nginx / Apache receives HTTP request        │
   │  Forwards to WSGI/ASGI application server   │
   └──────────────────────────────────────────────┘
                    ↓

3. DJANGO MIDDLEWARE (Request Phase)
   ┌──────────────────────────────────────────────┐
   │  SecurityMiddleware (HTTPS redirect, etc)    │
   │  SessionMiddleware (load session)            │
   │  AuthenticationMiddleware (load user)        │
   │  CsrfViewMiddleware (CSRF token check)       │
   │  ...and more...                              │
   └──────────────────────────────────────────────┘
                    ↓

4. URL ROUTING (urls.py)
   ┌──────────────────────────────────────────────┐
   │  URL: /blog/post/django-tutorial/            │
   │  Pattern: path('post/<slug:slug>/', ...)     │
   │  Match: Found!                            │
   │  View: PostDetailView                        │
   └──────────────────────────────────────────────┘
                    ↓

5. VIEW PROCESSING (views.py)
   ┌──────────────────────────────────────────────┐
   │  Execute PostDetailView.get()                │
   │  Check permissions                           │
   │  Query database: Post.objects.get(...)       │
   │  Get related data: post.comments.all()       │
   │  Pass to template context                    │
   └──────────────────────────────────────────────┘
                    ↓

6. DATABASE QUERIES (models.py)
   ┌──────────────────────────────────────────────┐
   │  SELECT * FROM blog_post WHERE slug = ...    │
   │  SELECT * FROM blog_comment WHERE post_id=.. │
   │  Results: Post object + Comments list        │
   └──────────────────────────────────────────────┘
                    ↓

7. TEMPLATE RENDERING (templates/)
   ┌──────────────────────────────────────────────┐
   │  Load: blog/post_detail.html                 │
   │  Render: {{ post.title }}, {% for comment %} │
   │  Auto-escape HTML for security               │
   │  Output: HTML string                         │
   └──────────────────────────────────────────────┘
                    ↓

8. DJANGO MIDDLEWARE (Response Phase)
   ┌──────────────────────────────────────────────┐
   │  Add security headers                        │
   │  Set session cookies                         │
   │  Compress response (gzip)                    │
   │  ...and more...                              │
   └──────────────────────────────────────────────┘
                    ↓

9. SEND RESPONSE TO CLIENT
   ┌──────────────────────────────────────────────┐
   │  Status: 200 OK                              │
   │  Headers: Content-Type, Cookies, etc         │
   │  Body: Rendered HTML                         │
   └──────────────────────────────────────────────┘
                    ↓

10. BROWSER DISPLAYS
   ┌──────────────────────────────────────────────┐
   │  Browser receives HTML                       │
   │  Parses HTML, CSS, JavaScript                │
   │  Displays page to user                       │
   └──────────────────────────────────────────────┘

Total Time: Typically 50-200ms
```

### 3.2 Detailed Middleware Pipeline

In [ ]:
# Middleware Processing Order

middleware_flow = """
settings.py MIDDLEWARE = [
    1. SecurityMiddleware          ← Executes first (request)
    2. SessionMiddleware
    3. AuthenticationMiddleware
    4. CsrfViewMiddleware
    5. XFrameOptionsMiddleware
    6. ContentSecurityPolicyMiddleware
]

REQUEST PHASE (top to bottom):
  1. SecurityMiddleware.process_request()
  2. SessionMiddleware.process_request()
  3. AuthenticationMiddleware.process_request()
  4. CsrfViewMiddleware.process_request()
  5. ...
  → URL Routing & View Execution

RESPONSE PHASE (bottom to top):
  6. XFrameOptionsMiddleware.process_response()
  5. ContentSecurityPolicyMiddleware.process_response()
  4. CsrfViewMiddleware.process_response()
  3. AuthenticationMiddleware.process_response()
  2. SessionMiddleware.process_response()
  1. SecurityMiddleware.process_response() ← Executes last

Example: CSRF Protection
────────────────────────
REQUEST:  Check CSRF token in POST data (CsrfViewMiddleware)
          If invalid → 403 Forbidden (request rejected)
          If valid → Continue to view

RESPONSE: Add CSRF token to outgoing response (for next form)
          Client gets token for next request
"""

print(middleware_flow)

---
## Module 4: Full-Fledged Django Application (Blog Platform)
---

### 4.1 Application Architecture Overview

```
┌────────────────────────────────────────────────────────────────┐
│                    BLOG PLATFORM ARCHITECTURE                   │
└────────────────────────────────────────────────────────────────┘

                         FRONTEND LAYER
            ┌──────────────────────────────────┐
            │  User Browser (HTML, CSS, JS)    │
            │  - View blog posts               │
            │  - Submit comments               │
            │  - User login                    │
            └────────────┬─────────────────────┘
                         │
                         │ HTTP Requests
                         ↓
                    WEB SERVER
            ┌──────────────────────────────────┐
            │  Nginx / Apache                  │
            │  - Handle static files           │
            │  - Load balance                  │
            │  - SSL/TLS termination           │
            └────────────┬─────────────────────┘
                         │
                         │ WSGI/ASGI
                         ↓
              APPLICATION LAYER
        ┌────────────────────────────────┐
        │   Django Application Server    │
        │   (Gunicorn/Daphne)            │
        │   - Multiple workers/processes │
        │   - Handle concurrent requests │
        └────────────┬───────────────────┘
                     │
    ┌────────────────┼────────────────┐
    │                │                │
    ↓                ↓                ↓
REQUEST        DJANGO CORE        CACHE LAYER
HANDLERS         LOGIC            ┌─────────────┐
┌────────────┐  ┌─────────────┐   │ Redis Cache │
│Middleware  │  │Views (MVT)  │   │- Session    │
│URL Router  │  │Models (ORM) │   │- Data cache │
│Error Hdlr  │  │Templates    │   │- Query      │
└────────────┘  │Auth/Perms   │   └─────────────┘
                └──────┬──────┘
                       │
        ┌──────────────┼──────────────┐
        │              │              │
        ↓              ↓              ↓
    DATABASE       FILE STORAGE    EXTERNAL SERVICES
   ┌──────────┐   ┌──────────┐    ┌────────────┐
   │PostgreSQL│   │AWS S3    │    │Email       │
   │MySQL     │   │Local Disk│    │Payment API │
   │SQLite    │   └──────────┘    │CDN         │
   └──────────┘                   └────────────┘
```

### 4.2 Blog Application - Complete Code Example

In [ ]:
# Blog Application - Models

code = """
# blog/models.py

from django.db import models
from django.contrib.auth.models import User
from django.utils.text import slugify

class Category(models.Model):
    name = models.CharField(max_length=100)
    slug = models.SlugField(unique=True)
    description = models.TextField(blank=True)
    created_at = models.DateTimeField(auto_now_add=True)
    
    class Meta:
        ordering = ['name']
    
    def __str__(self):
        return self.name

class BlogPost(models.Model):
    STATUS_CHOICES = [
        ('draft', 'Draft'),
        ('published', 'Published'),
        ('archived', 'Archived'),
    ]
    
    # Foreign Keys (Relationships)
    author = models.ForeignKey(User, on_delete=models.CASCADE, related_name='blog_posts')
    category = models.ForeignKey(Category, on_delete=models.SET_NULL, null=True, related_name='posts')
    
    # Content
    title = models.CharField(max_length=200)
    slug = models.SlugField(unique=True)
    content = models.TextField()
    excerpt = models.CharField(max_length=500, blank=True)
    featured_image = models.ImageField(upload_to='blog/%Y/%m/', blank=True)
    
    # Metadata
    status = models.CharField(max_length=20, choices=STATUS_CHOICES, default='draft')
    views_count = models.IntegerField(default=0)
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)
    published_at = models.DateTimeField(null=True, blank=True)
    
    # SEO
    meta_description = models.CharField(max_length=160, blank=True)
    tags = models.CharField(max_length=500, blank=True, help_text='Comma-separated')
    
    class Meta:
        ordering = ['-published_at']
        indexes = [
            models.Index(fields=['slug']),
            models.Index(fields=['author', '-published_at']),
            models.Index(fields=['status', '-published_at']),
        ]
    
    def __str__(self):
        return self.title
    
    def save(self, *args, **kwargs):
        if not self.slug:
            self.slug = slugify(self.title)
        super().save(*args, **kwargs)

class Comment(models.Model):
    post = models.ForeignKey(BlogPost, on_delete=models.CASCADE, related_name='comments')
    author = models.ForeignKey(User, on_delete=models.CASCADE)
    content = models.TextField()
    is_approved = models.BooleanField(default=False)
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)
    
    class Meta:
        ordering = ['-created_at']
    
    def __str__(self):
        return f'Comment by {self.author} on {self.post.title}'
"""

print("=" * 70)
print("BLOG APPLICATION - DATABASE MODELS")
print("=" * 70)
print(code)

In [ ]:
# Blog Application - Views

code = """
# blog/views.py

from django.shortcuts import render, get_object_or_404, redirect
from django.views.generic import ListView, DetailView, CreateView
from django.contrib.auth.mixins import LoginRequiredMixin
from django.db.models import Q, Count
from django.utils import timezone
from django.views.decorators.cache import cache_page
from django.core.paginator import Paginator
from .models import BlogPost, Comment, Category
from .forms import CommentForm

# List all published blog posts with pagination
class PostListView(ListView):
    model = BlogPost
    template_name = 'blog/post_list.html'
    context_object_name = 'posts'
    paginate_by = 10
    
    def get_queryset(self):
        # Only show published posts
        posts = BlogPost.objects.filter(status='published').order_by('-published_at')
        
        # Search functionality
        query = self.request.GET.get('q')
        if query:
            posts = posts.filter(
                Q(title__icontains=query) |
                Q(content__icontains=query) |
                Q(tags__icontains=query)
            )
        
        # Category filter
        category_slug = self.request.GET.get('category')
        if category_slug:
            posts = posts.filter(category__slug=category_slug)
        
        # Optimize queries
        return posts.select_related('author', 'category').prefetch_related('comments')
    
    def get_context_data(self, **kwargs):
        context = super().get_context_data(**kwargs)
        # Add categories for sidebar
        context['categories'] = Category.objects.annotate(
            post_count=Count('posts', filter=Q(posts__status='published'))
        ).filter(post_count__gt=0)
        context['search_query'] = self.request.GET.get('q', '')
        return context

# View single blog post with comments
class PostDetailView(DetailView):
    model = BlogPost
    template_name = 'blog/post_detail.html'
    slug_field = 'slug'
    context_object_name = 'post'
    
    def get_queryset(self):
        # Only show published posts to non-staff users
        queryset = BlogPost.objects.all()
        if not self.request.user.is_staff:
            queryset = queryset.filter(status='published')
        return queryset.select_related('author', 'category')
    
    def get_context_data(self, **kwargs):
        context = super().get_context_data(**kwargs)
        
        # Increment view count
        self.object.views_count += 1
        self.object.save(update_fields=['views_count'])
        
        # Get approved comments
        context['comments'] = self.object.comments.filter(is_approved=True)
        context['comment_form'] = CommentForm()
        
        # Related posts (same category)
        context['related_posts'] = BlogPost.objects.filter(
            category=self.object.category,
            status='published'
        ).exclude(id=self.object.id)[:3]
        
        return context

# Create new comment (authenticated users only)
class AddCommentView(LoginRequiredMixin, CreateView):
    model = Comment
    form_class = CommentForm
    login_url = 'login'
    
    def form_valid(self, form):
        comment = form.save(commit=False)
        comment.author = self.request.user
        comment.post = get_object_or_404(BlogPost, slug=self.kwargs['slug'])
        comment.save()
        return redirect('post_detail', slug=comment.post.slug)
"""

print("=" * 70)
print("BLOG APPLICATION - VIEWS (BUSINESS LOGIC)")
print("=" * 70)
print(code)

In [ ]:
# Blog Application - Admin Interface

code = """
# blog/admin.py

from django.contrib import admin
from django.utils.html import format_html
from .models import BlogPost, Category, Comment

@admin.register(Category)
class CategoryAdmin(admin.ModelAdmin):
    list_display = ('name', 'post_count')
    prepopulated_fields = {'slug': ('name',)}
    search_fields = ('name',)
    
    def post_count(self, obj):
        return obj.posts.count()
    post_count.short_description = 'Posts'

@admin.register(BlogPost)
class BlogPostAdmin(admin.ModelAdmin):
    list_display = ('title', 'author', 'status', 'category', 'views_count', 'published_at')
    list_filter = ('status', 'category', 'published_at', 'created_at')
    search_fields = ('title', 'content', 'slug')
    prepopulated_fields = {'slug': ('title',)}
    readonly_fields = ('created_at', 'updated_at', 'views_count')
    date_hierarchy = 'published_at'
    
    fieldsets = (
        ('Content', {
            'fields': ('title', 'slug', 'content', 'excerpt', 'featured_image')
        }),
        ('Metadata', {
            'fields': ('author', 'category', 'status', 'published_at')
        }),
        ('SEO', {
            'fields': ('meta_description', 'tags'),
            'classes': ('collapse',)
        }),
        ('Statistics', {
            'fields': ('views_count', 'created_at', 'updated_at'),
            'classes': ('collapse',)
        }),
    )
    
    def save_model(self, request, obj, form, change):
        if not change:  # New post
            obj.author = request.user
        super().save_model(request, obj, form, change)

@admin.register(Comment)
class CommentAdmin(admin.ModelAdmin):
    list_display = ('author', 'post', 'is_approved', 'created_at', 'preview')
    list_filter = ('is_approved', 'created_at')
    search_fields = ('author__username', 'content')
    readonly_fields = ('created_at', 'updated_at')
    
    fieldsets = (
        ('Comment Details', {
            'fields': ('post', 'author', 'content')
        }),
        ('Status', {
            'fields': ('is_approved',)
        }),
        ('Dates', {
            'fields': ('created_at', 'updated_at'),
            'classes': ('collapse',)
        }),
    )
    
    actions = ['approve_comments', 'reject_comments']
    
    def approve_comments(self, request, queryset):
        queryset.update(is_approved=True)
    
    def reject_comments(self, request, queryset):
        queryset.update(is_approved=False)
    
    def preview(self, obj):
        return obj.content[:100]
    preview.short_description = 'Preview'
"""

print("=" * 70)
print("BLOG APPLICATION - ADMIN CONFIGURATION")
print("=" * 70)
print(code)
print("\n Result: Auto-generated admin dashboard with:")
print("   - Create, read, update, delete posts")
print("   - Manage comments (approve/reject)")
print("   - Filter and search")
print("   - Bulk actions")
print("   - View statistics")

In [ ]:
# Blog Application - URL Routing

code = """
# blog/urls.py

from django.urls import path
from . import views

app_name = 'blog'

urlpatterns = [
    # List views
    path('', views.PostListView.as_view(), name='post_list'),
    path('category/<slug:slug>/', views.PostListView.as_view(), name='category_posts'),
    
    # Detail views
    path('post/<slug:slug>/', views.PostDetailView.as_view(), name='post_detail'),
    
    # Comment views
    path('post/<slug:slug>/comment/', views.AddCommentView.as_view(), name='add_comment'),
]

# main/urls.py (project-level)
from django.contrib import admin
from django.urls import path, include
from django.conf import settings
from django.conf.urls.static import static

urlpatterns = [
    path('admin/', admin.site.urls),
    path('blog/', include('blog.urls')),
    path('accounts/', include('django.contrib.auth.urls')),
]

# Serve media files in development
if settings.DEBUG:
    urlpatterns += static(settings.MEDIA_URL, document_root=settings.MEDIA_ROOT)
"""

print("=" * 70)
print("BLOG APPLICATION - URL ROUTING")
print("=" * 70)
print(code)

In [ ]:
# Blog Application - Template

code = """
<!-- blog/templates/blog/post_detail.html -->

{% extends 'base.html' %}
{% load static %}

{% block title %}{{ post.title }} - Blog{% endblock %}
{% block meta_description %}{{ post.meta_description|truncatewords:20 }}{% endblock %}

{% block content %}
<article class="blog-post">
    <!-- Post Header -->
    <header class="post-header">
        <h1>{{ post.title }}</h1>
        <div class="post-meta">
            <span>By <strong>{{ post.author.get_full_name }}</strong></span>
            <span>{{ post.published_at|date:"F d, Y" }}</span>
            <span>{{ post.views_count }} views</span>
            {% if post.category %}
                <span>in <a href="{% url 'blog:category_posts' post.category.slug %}">{{ post.category.name }}</a></span>
            {% endif %}
        </div>
    </header>
    
    <!-- Featured Image -->
    {% if post.featured_image %}
        <figure class="featured-image">
            <img src="{{ post.featured_image.url }}" alt="{{ post.title }}">
        </figure>
    {% endif %}
    
    <!-- Post Content -->
    <div class="post-content">
        {{ post.content|linebreaks }}
    </div>
    
    <!-- Tags -->
    {% if post.tags %}
        <div class="tags">
            {% for tag in post.tags|split:"," %}
                <span class="tag">#{{ tag|striptags }}</span>
            {% endfor %}
        </div>
    {% endif %}
    
    <!-- Related Posts -->
    {% if related_posts %}
        <section class="related-posts">
            <h3>Related Articles</h3>
            <div class="posts-grid">
                {% for related_post in related_posts %}
                    <div class="post-card">
                        <h4><a href="{% url 'blog:post_detail' related_post.slug %}">{{ related_post.title }}</a></h4>
                        <p>{{ related_post.excerpt|truncatewords:20 }}</p>
                    </div>
                {% endfor %}
            </div>
        </section>
    {% endif %}
</article>

<!-- Comments Section -->
<section class="comments-section">
    <h3>Comments ({{ post.comments.count }})</h3>
    
    <!-- Display Comments -->
    <div class="comments-list">
        {% for comment in comments %}
            <div class="comment" id="comment-{{ comment.id }}">
                <strong>{{ comment.author.get_full_name }}</strong>
                <span class="comment-date">{{ comment.created_at|timesince }} ago</span>
                <p>{{ comment.content|escape }}</p>
            </div>
        {% empty %}
            <p>No comments yet. Be the first to comment!</p>
        {% endfor %}
    </div>
    
    <!-- Add Comment Form -->
    {% if user.is_authenticated %}
        <form method="post" action="{% url 'blog:add_comment' post.slug %}" class="comment-form">
            {% csrf_token %}
            {{ comment_form.as_p }}
            <button type="submit" class="btn btn-primary">Post Comment</button>
        </form>
    {% else %}
        <p><a href="{% url 'login' %}">Login</a> to post a comment.</p>
    {% endif %}
</section>
{% endblock %}
"""

print("=" * 70)
print("BLOG APPLICATION - TEMPLATE (PRESENTATION)")
print("=" * 70)
print(code)

### 4.3 How Full Application Works - Data Flow

```
┌────────────────────────────────────────────────────────────────┐
│           BLOG APPLICATION - COMPLETE DATA FLOW                │
└────────────────────────────────────────────────────────────────┘

SCENARIO: User visits /blog/post/django-tutorial/

STEP 1: URL MATCHING
────────────────────
URL: /blog/post/django-tutorial/
Pattern: path('post/<slug:slug>/', views.PostDetailView.as_view())
Match:  slug = 'django-tutorial'
View: PostDetailView

STEP 2: VIEW EXECUTION
──────────────────────
PostDetailView.get_object():
  → Query: BlogPost.objects.get(slug='django-tutorial')
  → SQL: SELECT * FROM blog_blogpost WHERE slug = 'django-tutorial'
  → Result: BlogPost object with id=5

PostDetailView.get_context_data():
  → comments = post.comments.filter(is_approved=True)
  → SQL: SELECT * FROM blog_comment WHERE post_id=5 AND is_approved=1
  → related_posts = BlogPost.objects.filter(category=post.category)...
  → context = {
      'post': BlogPost(id=5, title='Django...', ...),
      'comments': [Comment(...), Comment(...), ...],
      'related_posts': [BlogPost(...), ...],
    }

STEP 3: TEMPLATE RENDERING
───────────────────────────
Template: blog/post_detail.html
Variables: {{ post.title }}, {{ post.content }}, {% for comment in comments %}

Rendering Process:
  <h1>{{ post.title }}</h1>
     ↓ Lookup 'post' in context
     ↓ Get 'title' attribute
     ↓ Auto-escape HTML (security)
  <h1>Django Framework Guide</h1>
  
  {% for comment in comments %}
     ↓ Iterate 3 comments
     ↓ Render block 3 times
     ↓ Escape comment.content

STEP 4: RESPONSE CONSTRUCTION
──────────────────────────────
Middleware (Response Phase):
  ✓ Add security headers (X-Frame-Options, etc)
  ✓ Set-Cookie: sessionid=xyz123; Path=/; HttpOnly
  ✓ Add CSRF token for next form submission
  ✓ Compress response (gzip)

HTTP Response:
  Status: 200 OK
  Headers:
    Content-Type: text/html; charset=utf-8
    Content-Length: 12456
    Set-Cookie: sessionid=abc123; ...
    X-Frame-Options: DENY
  Body: <html>...</html> (rendered HTML)

STEP 5: BROWSER DISPLAY
───────────────────────
Browser receives response
  → Parse HTML
  → Load CSS files
  → Execute JavaScript
  → Display page to user
  → Store sessionid cookie

Total Time: ~100-200ms

DATABASE INTERACTION SUMMARY:
─────────────────────────────
Query 1: SELECT * FROM blog_blogpost WHERE slug = 'django-tutorial'
Query 2: SELECT * FROM blog_comment WHERE post_id = 5 AND is_approved = 1
Query 3: SELECT * FROM blog_blogpost WHERE category_id = 2 AND status = 'published'
Query 4: UPDATE blog_blogpost SET views_count = 42 WHERE id = 5

 Optimizations Applied:
   - select_related('author', 'category') → Reduce queries
   - prefetch_related('comments') → Single batch query
   - update_fields=['views_count'] → Only update that field
   - Indexes on (slug), (author, published_at), (status, published_at)
```

---
## Module 5: Django REST Framework (DRF) - API Development
---

### 5.1 What is Django REST Framework?

**Definition:** A powerful toolkit for building REST APIs in Django.

**Key Features:**
- Automatic serialization (Model → JSON)
- Built-in authentication & permissions
- API views and viewsets
- Filtering, searching, pagination
- Automatic API documentation
- Request/response validation

### 5.2 Django vs Django REST Framework

```
┌─────────────────────────────────────────────────────────────┐
│              DJANGO vs DJANGO REST FRAMEWORK                │
└─────────────────────────────────────────────────────────────┘

                    TRADITIONAL DJANGO
        (Renders HTML for Browser Consumption)
                           ↓
                    ┌──────────────┐
                    │   Browser    │
                    │              │
                    │   GET /blog/ │
                    └────────┬─────┘
                             │
                    ┌────────▼──────────┐
                    │  Django View      │
                    │  - Query DB       │
                    │  - Render Template│
                    │  - Return HTML    │
                    └────────┬──────────┘
                             │
                ┌────────────▼───────────────┐
                │   HTTP Response (HTML)     │
                │                            │
                │ <html>                     │
                │   <h1>Blog Posts</h1>      │
                │   <div>Post 1</div>        │
                │   <div>Post 2</div>        │
                │ </html>                    │
                └────────────────────────────┘
                             │
                    ┌────────▼──────────┐
                    │  Browser renders  │
                    │  HTML + displays  │
                    └───────────────────┘

═════════════════════════════════════════════════════════════

              DJANGO REST FRAMEWORK
    (Returns JSON for Mobile/Frontend/API Consumption)
                           ↓
        ┌─────────────────┬──────────────────┐
        │                 │                  │
    ┌───▼────┐      ┌─────▼──────┐     ┌────▼────┐
    │Mobile  │      │JavaScript  │     │External │
    │App     │      │Frontend    │     │Service  │
    │        │      │(React/Vue) │     │(Alexa)  │
    └───┬────┘      └─────┬──────┘     └────┬────┘
        │                 │                 │
        │  GET /api/posts │                 │
        │ (JSON request)  │                 │
        └─────────────────┼─────────────────┘
                          │
                 ┌────────▼───────────┐
                 │ DRF Serializer     │
                 │ - Model → JSON     │
                 │ - Validation       │
                 │ - Permissions      │
                 └────────┬───────────┘
                          │
                 ┌────────▼──────────┐
                 │  DRF ViewSet      │
                 │  - Query DB       │
                 │  - Return JSON    │
                 │  - Handle CRUD    │
                 └────────┬──────────┘
                          │
         ┌────────────────▼──────────────────┐
         │   HTTP Response (JSON)            │
         │                                   │
         │ {                                 │
         │   "count": 42,                    │
         │   "results": [                    │
         │     {                             │
         │       "id": 1,                    │
         │       "title": "Django Guide",    │
         │       "author": "John",           │
         │       "created_at": "2024-01-01"  │
         │     },                            │
         │     ...                           │
         │   ]                               │
         │ }                                 │
         └───────────────┬────────────────────┘
                         │
        ┌────────────────┼──────────────────┐
        │                │                  │
    ┌───▼────┐      ┌────▼─────┐     ┌─────▼───┐
    │Mobile  │      │JavaScript│     │External │
    │App     │      │App       │     │Service  │
    │parses  │      │parses    │     │parses   │
    │JSON    │      │JSON      │     │JSON     │
    │display │      │render    │     │process  │
    └────────┘      └──────────┘     └─────────┘
```

### 5.3 Key Differences Summary

| Aspect | Traditional Django | Django REST Framework |
|--------|-------------------|----------------------|
| **Output Format** | HTML | JSON |
| **Client Type** | Web Browser | Mobile, Frontend, APIs |
| **Serialization** | Template rendering | Serializers (Model → JSON) |
| **Request Handling** | Form data | JSON payloads |
| **Authentication** | Session-based | Token/JWT-based |
| **Permissions** | View-level | View + Object-level |
| **Response** | HTML page | JSON object |
| **Status Codes** | 200 OK, 404, 500 | 200, 201, 400, 401, 403, 404 |
| **Documentation** | Manual | Auto-generated |
| **Testing** | Browser/Selenium | API testing tools |
| **Content-Type** | text/html | application/json |
| **CSRF Token** | Required in forms | Not needed (different auth) |
| **Pagination** | Manual queryset slicing | Built-in pagination classes |
| **Filtering** | Manual query parameters | FilterSet classes |
| **Versioning** | Not applicable | API versioning support |

### 5.4 Building Blog API with Django REST Framework

In [ ]:
# Blog API - Serializers

code = """
# blog/serializers.py

from rest_framework import serializers
from .models import BlogPost, Comment, Category

class CategorySerializer(serializers.ModelSerializer):
    class Meta:
        model = Category
        fields = ['id', 'name', 'slug', 'description']

class CommentSerializer(serializers.ModelSerializer):
    author_name = serializers.CharField(source='author.get_full_name', read_only=True)
    
    class Meta:
        model = Comment
        fields = ['id', 'author_name', 'content', 'is_approved', 'created_at']
        read_only_fields = ['created_at']

class BlogPostSerializer(serializers.ModelSerializer):
    # Nested serializers
    author_name = serializers.CharField(source='author.get_full_name', read_only=True)
    category = CategorySerializer(read_only=True)
    comments = CommentSerializer(many=True, read_only=True)
    comment_count = serializers.SerializerMethodField()
    
    class Meta:
        model = BlogPost
        fields = [
            'id', 'title', 'slug', 'content', 'excerpt',
            'author_name', 'category', 'status',
            'featured_image', 'views_count',
            'created_at', 'published_at',
            'comments', 'comment_count'
        ]
        read_only_fields = ['created_at', 'published_at', 'views_count']
    
    def get_comment_count(self, obj):
        return obj.comments.filter(is_approved=True).count()

class BlogPostCreateSerializer(serializers.ModelSerializer):
    '''Serializer for creating posts (different from read)'''
    class Meta:
        model = BlogPost
        fields = ['title', 'content', 'excerpt', 'category', 'featured_image', 'meta_description', 'tags']
"""

print("=" * 70)
print("BLOG API - SERIALIZERS (Model → JSON)")
print("=" * 70)
print(code)

In [ ]:
# Blog API - Views & ViewSets

code = """
# blog/views.py (API)

from rest_framework import viewsets, status, filters
from rest_framework.decorators import action, api_view, permission_classes
from rest_framework.response import Response
from rest_framework.permissions import IsAuthenticated, IsAuthenticatedOrReadOnly
from rest_framework.pagination import PageNumberPagination
from django_filters.rest_framework import DjangoFilterBackend
from .models import BlogPost, Comment, Category
from .serializers import BlogPostSerializer, CommentSerializer, CategorySerializer

class StandardPagination(PageNumberPagination):
    page_size = 10
    page_size_query_param = 'page_size'
    max_page_size = 100

class BlogPostViewSet(viewsets.ModelViewSet):
    '''API ViewSet for BlogPost CRUD operations'''
    queryset = BlogPost.objects.filter(status='published')
    serializer_class = BlogPostSerializer
    permission_classes = [IsAuthenticatedOrReadOnly]
    pagination_class = StandardPagination
    filter_backends = [DjangoFilterBackend, filters.SearchFilter, filters.OrderingFilter]
    filterset_fields = ['category', 'author']
    search_fields = ['title', 'content', 'tags']
    ordering_fields = ['created_at', 'views_count']
    ordering = ['-published_at']
    lookup_field = 'slug'
    
    def get_queryset(self):
        '''Optimize queries with select_related'''
        queryset = super().get_queryset()
        return queryset.select_related('author', 'category').prefetch_related('comments')
    
    def perform_create(self, serializer):
        '''Set current user as author'''
        serializer.save(author=self.request.user)
    
    @action(detail=True, methods=['post'], permission_classes=[IsAuthenticated])
    def add_comment(self, request, slug=None):
        '''Custom action to add comment to post'''
        post = self.get_object()
        serializer = CommentSerializer(data=request.data)
        if serializer.is_valid():
            serializer.save(post=post, author=request.user)
            return Response(serializer.data, status=status.HTTP_201_CREATED)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)
    
    @action(detail=True, methods=['get'])
    def related(self, request, slug=None):
        '''Get related posts'''
        post = self.get_object()
        related = post.category.posts.exclude(id=post.id)[:3]
        serializer = self.get_serializer(related, many=True)
        return Response(serializer.data)

class CommentViewSet(viewsets.ModelViewSet):
    '''API ViewSet for Comment CRUD operations'''
    queryset = Comment.objects.filter(is_approved=True)
    serializer_class = CommentSerializer
    permission_classes = [IsAuthenticatedOrReadOnly]
    filter_backends = [DjangoFilterBackend]
    filterset_fields = ['post']

class CategoryViewSet(viewsets.ReadOnlyModelViewSet):
    '''API ViewSet for reading categories (read-only)'''
    queryset = Category.objects.all()
    serializer_class = CategorySerializer
    lookup_field = 'slug'
"""

print("=" * 70)
print("BLOG API - VIEWSETS (API ENDPOINTS)")
print("=" * 70)
print(code)

In [ ]:
# Blog API - URL Configuration

code = """
# blog/urls.py (API routes)

from django.urls import path, include
from rest_framework.routers import DefaultRouter
from . import views

# Create router and register viewsets
router = DefaultRouter()
router.register(r'posts', views.BlogPostViewSet, basename='blogpost')
router.register(r'comments', views.CommentViewSet, basename='comment')
router.register(r'categories', views.CategoryViewSet, basename='category')

# Router automatically generates URLs:
# GET    /api/posts/               - List all posts
# POST   /api/posts/               - Create new post
# GET    /api/posts/{slug}/        - Get specific post
# PUT    /api/posts/{slug}/        - Update post
# DELETE /api/posts/{slug}/        - Delete post
# POST   /api/posts/{slug}/add_comment/  - Add comment
# GET    /api/posts/{slug}/related/    - Get related posts

urlpatterns = [
    path('api/', include(router.urls)),
    path('api-auth/', include('rest_framework.urls')),
]

# In main project/urls.py:
urlpatterns = [
    path('blog/', include('blog.urls')),
]
"""

print("=" * 70)
print("BLOG API - URL ROUTING (AUTOMATIC ENDPOINT GENERATION)")
print("=" * 70)
print(code)

### 5.5 API Request-Response Example

In [ ]:
# API Request-Response Cycle

example = """
═══════════════════════════════════════════════════════════════
REQUEST: Get all blog posts
═══════════════════════════════════════════════════════════════

GET /api/posts/?category=django&search=tutorial&page=1
Authorization: Token abc123xyz789
Content-Type: application/json


═══════════════════════════════════════════════════════════════
PROCESSING IN DRF
═══════════════════════════════════════════════════════════════

1. Permission Check:
   → IsAuthenticatedOrReadOnly: GET allowed (no auth needed)
   Permission granted

2. Filter & Search:
   → category__slug='django' (from DjangoFilterBackend)
   → title__icontains='tutorial' OR tags__icontains='tutorial'
   → Query: Posts where category=django AND (title or tags contain tutorial)

3. Ordering:
   → Order by: -published_at (newest first)

4. Pagination:
   → page_size: 10
   → page: 1
   → Skip 0, Take 10

5. Query Optimization:
   → select_related('author', 'category')
   → prefetch_related('comments')
   → Execute 2-3 SQL queries (not N+1)

6. Serialization:
   → Convert BlogPost objects to JSON
   → Serialize nested: author_name, category, comments
   → Validate field values


═══════════════════════════════════════════════════════════════
RESPONSE: 200 OK
═══════════════════════════════════════════════════════════════

HTTP/1.1 200 OK
Content-Type: application/json
Content-Length: 4567

{
  "count": 42,
  "next": "http://api.example.com/api/posts/?page=2",
  "previous": null,
  "results": [
    {
      "id": 5,
      "title": "Django REST Framework Tutorial",
      "slug": "django-rest-framework-tutorial",
      "content": "Building APIs with Django...",
      "excerpt": "Learn how to build REST APIs...",
      "author_name": "John Doe",
      "category": {
        "id": 1,
        "name": "Django",
        "slug": "django",
        "description": "Django web framework tutorials"
      },
      "status": "published",
      "featured_image": "/media/blog/2024/01/django-api.jpg",
      "views_count": 1523,
      "created_at": "2024-01-15T10:30:00Z",
      "published_at": "2024-01-15T12:00:00Z",
      "comment_count": 5,
      "comments": [
        {
          "id": 12,
          "author_name": "Jane Smith",
          "content": "Great tutorial! Very helpful.",
          "is_approved": true,
          "created_at": "2024-01-16T09:15:00Z"
        }
      ]
    }
  ]
}


═══════════════════════════════════════════════════════════════
KEY BENEFITS OF DRF
═══════════════════════════════════════════════════════════════

Serialization: Model → JSON automatically
Validation: Built-in data validation
Authentication: Multiple auth methods (Token, JWT, OAuth)
Permissions: View-level & object-level permissions
Filtering: SearchFilter, DjangoFilterBackend
Pagination: Automatic with configurable page sizes
Documentation: Auto-generated browsable API
Error Handling: Consistent error responses
Status Codes: Appropriate HTTP status codes
Content Negotiation: JSON, XML, YAML formats
"""

print(example)

---
## Module 6: Summary & Next Steps
---

### 6.1 Complete Learning Path

**Phase 1: Fundamentals (1-2 weeks)**
- MTV Architecture
- Models, Views, Templates
- URL Routing
- Admin Interface
- Migrations

**Phase 2: Intermediate (2-4 weeks)**
- Authentication & Permissions
- Forms & Validation
- QuerySet Optimization
- Class-Based Views
- Testing

**Phase 3: Advanced (4-8 weeks)**
- Django REST Framework
- Serializers & ViewSets
- Token/JWT Authentication
- Background Tasks (Celery)
- Caching (Redis)
- Signals & Middleware

**Phase 4: Deployment (Ongoing)**
- WSGI/ASGI Configuration
- Docker Containerization
- Production Settings
- Load Balancing
- Monitoring & Logging

### 6.2 Key Concepts Summary

| Concept | Purpose | Example |
|---------|---------|----------|
| **Models** | Database schema | BlogPost, Comment |
| **Views** | Business logic | PostDetailView, PostListView |
| **Templates** | HTML rendering | blog/post_detail.html |
| **URLs** | Route mapping | path('post/<slug>/', view) |
| **Admin** | CRUD interface | @admin.register |
| **Forms** | Input validation | CommentForm |
| **Serializers** | Object → JSON | BlogPostSerializer |
| **ViewSets** | API endpoints | BlogPostViewSet |
| **Middleware** | Request processing | CsrfViewMiddleware |
| **Signals** | Event system | post_save, post_delete |
| **WSGI** | Sync server interface | Gunicorn |
| **ASGI** | Async server interface | Daphne |

### 6.3 Resources for Further Learning

**Official Documentation**
- Django Docs: https://docs.djangoproject.com/
- Django REST Framework: https://www.django-rest-framework.org/

**Tutorials & Guides**
- Real Python Django
- MDN Django Tutorial
- Django for Beginners (Free)
- Two Scoops of Django

**Practice Projects**
1. Blog Platform (what we built here)
2. E-Commerce Store
3. Social Media Platform
4. Task Management App
5. Real-Time Chat Application

**Tools & Libraries**
- django-rest-framework
- celery (background tasks)
- redis (caching)
- pytest-django (testing)
- django-cors-headers
- django-filter
- graphql-core (GraphQL support)